# Vertex Batch Jobs

- Setting up Vertex Batch Jobs on a sample of data from BigQuery Tables
- 2 Separate Jobs: one for extraction of information for articles, the other is for daily summaries

In [ ]:
import ast
import google
import json
import re
import pandas as pd

from google import genai
from google.cloud import bigquery
from google.cloud import bigquery_storage_v1
from google.genai.types import CreateBatchJobConfig, HttpOptions
from vertexai.generative_models import GenerativeModel

# Defines

## GCP

In [ ]:
_, PROJECT_ID = google.auth.default()
REGION = "europe-west4"
BUCKET_NAME = "fin-news"
RAW_DATA_DIR = "raw/articles"

# BigQuery params
DATASET_ID = BUCKET_NAME.replace("-", "_")
TABLE_RAW_ID = "articles-raw"
TABLE_JOB_INPUT_ID = "articles-batch-job-input"
TABLE_JOB_OUTPUT_ID = "articles-batch-job-output"
TABLE_PROCESSED_ID = "articles-processed"

FULL_TABLE_RAW_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_RAW_ID}"
FULL_TABLE_INPUT_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_JOB_INPUT_ID}"
FULL_TABLE_OUTPUT_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_JOB_OUTPUT_ID}"
FULL_TABLE_PROCESSED_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_PROCESSED_ID}"

# VertexAI Params
MODEL_ID = "gemini-2.0-flash-001"

In [ ]:
bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    location=REGION
)

bqstorage = bigquery_storage_v1.BigQueryReadClient()

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset = bigquery_client.create_dataset(dataset_ref, exists_ok=True)

## Constants

In [ ]:
system_prompt = \
    f"You are an information extraction assistant. You are working with the financial news articles and " \
    f"opinions pieces: they cover news about business developments, macroeconomic developments, and " \
    f"personal views which, by their nature, do not have to be factual. " \
    f"The articles were extracted online from a range of open-source channels. Not all " \
    f"articles you are working with are factual - some intend to act as an ad to " \
    f"third-party services or express and opinion which is hard to factually verify."

## Methods

In [ ]:
def make_user_prompt(
    symbol: str,
    name: str,
    article: str
):
    user_prompt = \
    f"Determine if the following article mentions the company with the ticker symbol '{symbol}' ({name}). " \
    f"If yes - provide the summary of the section of the article which covers the company. The summary " \
    f"should be based on facts (not opinions), and it must exclude any trivial turns of phrase. " \
    f"Summaries must be at most 1500 characters long. \n\n" \
    f"""Classify the nature of the mention ('piece_type' field):

    - use 'factual' if the portion primarily covers events which happened to the company, or are relevant to the company
    - use 'market' if the portion primarily describes the recent changes in the company's share price, or changes in share prices of other companies / indices
    - use 'opinion' if the portion describes what market participants / investment firms / commentators think about the company, whether they hold it in their portfolio or suggest how the stock might perform in the future. E.g. Zachs Consensus Estimate ("Zachs") would be an 'opinion' piece

Exclude any parts of the text which promote any third-party equity research organisations.

Return a single JSON object:
{{
  'mentioned': true/false,
  'mention_type': 'direct' / 'indirect' / null,
  'extracted_text': '...' or null,
  'piece_type': 'factual' / 'market' / 'opinion' / null
}}

Article:
{article}
"""

    return user_prompt

In [ ]:
def make_request(row):
    user_prompt = make_user_prompt(
        row['symbol'],
        row['name'],
        row['article']
    )
    
    out = {
        "contents": [
            {
                "role": "user", 
                "parts": [{"text": user_prompt}]
            }
          ],
        "system_instruction": {
            "parts": [{"text": system_prompt}]
        },
        "generationConfig": {
            "temperature": 0
        }
    }

    return out

## Raw Data

In [ ]:
df_raw = pd.read_parquet('gs://fin-news/raw/articles/')

# Raw Data to BigQuery 'Request Job' table

Uploading a sample of articles from `df_raw` to a BigQuery table + a new 'request' column which will be used by the Batch Job in the next step.

In [ ]:
symbols = ['CAT', 'GOOG', 'GS', 'NKE', 'NVDA', 'V']

company_names = {
    'CAT': 'Caterpillar',
    'GOOG': 'Google',
    'GS': 'Goldman Sachs',
    'NKE': 'Nike',
    'NVDA': 'NVIDIA',
    'V': 'Visa'
}

In [ ]:
df = df_raw.loc[df_raw['symbol'].isin(symbols)].reset_index(drop=True).copy()

df['name'] = df['symbol'].replace(company_names)

In [ ]:
df['request'] = df.apply(
    lambda x: make_request(x),
    axis=1
)

df["date"] = df["date"].dt.strftime("%Y-%m-%dT%H:%M:%S")

## Uploading Raw Articles to BigQuery

In [ ]:
bigquery_client.load_table_from_dataframe(
    dataframe=df.drop(columns=['name', 'request']),
    destination=FULL_TABLE_RAW_ID
)

## Uploading Requests Table to BigQuery

In [ ]:
# Defining a load job with JSON schema
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("date", "TIMESTAMP"),
        bigquery.SchemaField("symbol", "STRING"),
        bigquery.SchemaField("article", "STRING"),
        bigquery.SchemaField("name", "STRING"),
        bigquery.SchemaField("request", "JSON") 
    ],
    write_disposition="WRITE_TRUNCATE"
)

job = bigquery_client.load_table_from_json(
    json_rows=df.to_dict(orient="records"),
    destination=FULL_TABLE_INPUT_ID,
    job_config=job_config
)

In [ ]:
job.result()

## Testing the Upload

In [ ]:
# Testing the output
sql = f"SELECT * FROM `{FULL_TABLE_INPUT_ID}`"

# submit & materialize via Storage API
job = bigquery_client.query(sql)
df_out = job.to_dataframe(bqstorage_client=bqstorage)
df_out.head()

# Article Information Extraction Job

Running a batch job on `request` column of 'Request Job' table from the last step; here we are getting the LLM to provide article summaries.

## Prompt Setup

Checking what the current prompt will do.

In [ ]:
model = GenerativeModel(
    model_name=MODEL_ID,
    generation_config={"temperature": 0.0}
)

In [ ]:
for i in range(0, 10):

    cur_row = df.iloc[i]

    prompt = make_request(cur_row)
    prompt = prompt['contents'][0]['parts'][0]['text']

    response = model.generate_content(
        prompt,
        generation_config={
            "response_mime_type": "application/json",
            "temperature": 0.0
        }
    )

    print(response.text)

## Job Setup

In [ ]:
job_time_suffix = pd.Timestamp.now().strftime("%Y-%M-%dT%H%M%S")

In [ ]:
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=REGION,
    http_options=HttpOptions(api_version="v1")
)

In [ ]:
input_uri = f"bq://{FULL_TABLE_INPUT_ID}"
output_uri = f"bq://{FULL_TABLE_OUTPUT_ID}"

In [ ]:
input_uri, output_uri

In [ ]:
job = client.batches.create(
    model=MODEL_ID,
    src=input_uri,
    config=CreateBatchJobConfig(
        display_name=f"batch-job-article-test-{job_time_suffix}",
        dest=output_uri
    )
)

## Results

### Fetching Job Results

In [ ]:
def json_to_series(x):
    out = pd.Series(
        {
            'mentioned': False,
            'mention_type': None,
            'extracted_text': None,
            'piece_type': None
        }
    )
    
    clean = re.sub(r'^```json[\s\\n]*|[\s\\n]*```$', '', x, flags=re.S)

    clean = re.sub(
        r'("extracted_text"\s*:\s*")([\s\S]*?)(",\s*\n\s*"piece_type")',
        lambda m: m.group(1) + m.group(2).replace('"', r'\"') + m.group(3),
        clean,
        flags=re.S
    )
    
    # 2) try real JSON first
    try:
        return pd.Series(json.loads(clean))
    except json.JSONDecodeError:
        try:
            # 3) fallback: convert JSON literals to Python ones, then ast.parse
            py = (
                clean
                .replace('true',  'True')
                .replace('false', 'False')
                .replace('null',  'None')
            )
            return pd.Series(ast.literal_eval(py))
        except Exception as e:
            return out

In [ ]:
# Fetching job output query
sql = f"SELECT * FROM `{FULL_TABLE_OUTPUT_ID}`"

# submit & materialize via Storage API
job = bigquery_client.query(sql)

In [ ]:
df_out = job.to_dataframe(bqstorage_client=bqstorage)

In [ ]:
df_out['parsed'] = df_out['response'].apply(json.loads)
# extract all “text” fields under content→parts (you can adjust [0] if you know there’s always one part)
df_out['texts'] = df_out['parsed'].apply(
    lambda j: j['candidates'][0]['content']['parts'][0]['text']
)

In [ ]:
df_out_text = df_out['texts'].apply(
    json_to_series
)

### Saving to BigQuery

In [ ]:
df_out = df_out[['date', 'symbol', 'article', 'name', 'processed_time']].join(df_out_text)
df_out = df_out.loc[df_out['mentioned']]
df_out = df_out.drop(columns=['mentioned'])
df_out = df_out.sort_values(by=['symbol', 'date'], ignore_index=True)

In [ ]:
bigquery_client.load_table_from_dataframe(
    dataframe=df_out,
    destination=FULL_TABLE_PROCESSED_ID
)

# Daily Summaries Job

Running a batch job that creates daily summaries out of news extracts created in the previous step. We are getting the LLM to provide article summaries.

In [ ]:
TABLE_JOB_INPUT_ID = "articles-daily-batch-job-input"
TABLE_JOB_OUTPUT_ID = "articles-daily-batch-job-output"
TABLE_PROCESSED_ID = "articles-daily-processed"

FULL_TABLE_INPUT_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_JOB_INPUT_ID}"
FULL_TABLE_OUTPUT_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_JOB_OUTPUT_ID}"

## Methods

In [ ]:
system_prompt = \
    f"You are an information summarisation assistant. You are working with pre-processed " \
    f"extracts of financial news articles and opinion pieces about large publicly traded " \
    f"companies. The original articles were pre-processed by another LLM."

In [ ]:
def make_request(df_cur):
    symbol = df_cur['symbol'].unique()[0]
    name = df_cur['name'].unique()[0]
    articles = df_cur[['extracted_text', 'piece_type']].to_json(orient='records')
    
    user_prompt = f"""
Create an executive daily summary of the article summaries presented about the company 
{name} (symbol {symbol}) as a json below. Your summary should contain three sections:

- 'News': what factual news stories were reported about the company, its market, competitors 
or trends in the economy or policy ('macro' factors) which affect the company
- 'Market': what was the qualitative summary of the day's price moves of the company? Was it
up or down, and were the moves big (over 10%), medium size (5%) or small?
- 'Opinion': summary of everything else

    
Articles:
{articles}
"""
    
    out = {
        "contents": [
            {
                "role": "user", 
                "parts": [{"text": user_prompt}]
            }
          ],
        "system_instruction": {
            "parts": [{"text": system_prompt}]
        },
        "generationConfig": {
            "temperature": 0
        }
    }

    return out

## Uploading Requests Table to BigQuery

In [ ]:
# Fetching job output query
sql = f"SELECT * FROM `{FULL_TABLE_PROCESSED_ID}`"

# submit & materialize via Storage API
job = bigquery_client.query(sql)
df_raw = job.to_dataframe(bqstorage_client=bqstorage)

In [ ]:
df = df_raw.copy()
df['date'] = df['date'].dt.floor('D')
df = df.sort_values(by=['symbol', 'date'], ignore_index=True)

In [ ]:
df_agg = df.groupby(
    ['symbol', 'date'],
)[df.columns].apply(
    make_request
).to_frame('request')

df_agg = df_agg.reset_index()

In [ ]:
df_agg["date"] = df_agg["date"].dt.strftime("%Y-%m-%dT%H:%M:%S")

In [ ]:
# Define the load job with JSON schema
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("symbol", "STRING"),
        bigquery.SchemaField("date", "TIMESTAMP"),
        bigquery.SchemaField("request", "JSON") 
    ],
    write_disposition="WRITE_TRUNCATE"
)

job = bigquery_client.load_table_from_json(
    json_rows=df_agg.to_dict(orient="records"),
    destination=FULL_TABLE_INPUT_ID,
    job_config=job_config
)

In [ ]:
job.result()

## Batch Prediction Job

In [ ]:
job_time_suffix = pd.Timestamp.now().strftime("%Y-%M-%dT%H%M%S")

In [ ]:
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=REGION,
    http_options=HttpOptions(api_version="v1")
)

In [ ]:
input_uri = f"bq://{FULL_TABLE_INPUT_ID}"
output_uri = f"bq://{FULL_TABLE_OUTPUT_ID}"

In [ ]:
job = client.batches.create(
    model=MODEL_ID,
    src=input_uri,
    config=CreateBatchJobConfig(
        display_name=f"batch-job-articles-daily-{job_time_suffix}",
        dest=output_uri
    )
)

## Results

In [ ]:
# Fetching job output query
sql = f"SELECT * FROM `{FULL_TABLE_OUTPUT_ID}`"

# submit & materialize via Storage API
job = bigquery_client.query(sql)

In [ ]:
df_out = job.to_dataframe(bqstorage_client=bqstorage)

In [ ]:
df_out['parsed'] = df_out['response'].apply(json.loads)
# extract all “text” fields under content→parts (you can adjust [0] if you know there’s always one part)
df_out['texts'] = df_out['parsed'].apply(
    lambda j: j['candidates'][0]['content']['parts'][0]['text']
)

In [ ]:
clean = re.sub(
    r'^```json[\{\s\\n]*|[\s\\n]*```$', '', 
    df_out['texts'].iloc[3326], 
    flags=re.S
)

In [ ]:
print(clean)